![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

## Quiver CNBC Trading Research

This notebook studies whether CNBC buy-recommendation count helps explain next open-to-open returns.

In [ ]:
qb = QuantBook()
# Daily bars will have an end_time that matches the following midnight.
qb.settings.daily_precise_end_time = False

### Build a CNBC Universe

Select assets with 3 or more BUY recommendations from CNBC commentators, then inspect the returned universe history.

In [ ]:
def select_assets(data: List[QuiverCNBCsUniverse]) -> List[Symbol]:
    # Group raw CNBC opinions by ticker so we can score each name.
    cnbc_by_symbol: dict[Symbol, list[QuiverCNBCsUniverse]] = {}
    for d in data:
        cnbc_by_symbol.setdefault(d.symbol, []).append(d)
    # Keep names with 3+ BUY recommendations to filter out noise.
    return [s for s, ds in cnbc_by_symbol.items()
            if sum(1 for d in ds if d.direction == OrderDirection.BUY) >= 3]

# Add the Quiver CNBC universe.
universe = qb.add_universe(QuiverCNBCsUniverse, select_assets)
# Request universe history of the last 120 days.
universe_history = qb.universe_history(universe, qb.time - timedelta(120), qb.time - timedelta(1), flatten=True)
# Print the returned shape and columns.
print(f"Shape: {universe_history.shape}")
print(f"Columns: {list(universe_history.columns)}")
universe_history.head()

### Universe Diagnostics

Aggregates the raw, per-mention CNBC commentary rows into a single summary row per `(time, symbol)`, describes data factor distributions of `buycount` and `totalmentions`, and visualizes how the unique asset footprint expands chronologically.

In [ ]:
# Create a column where 1 represents a BUY recommendation.
universe_history['isbuy'] = (universe_history['direction'].astype(str).str.lower() == 'buy').astype(int)
# Extract time and symbol arrays.
by_time = universe_history['time'].values if 'time' in universe_history.columns else universe_history.index.get_level_values(0)
by_symbol = universe_history['symbol'].values if 'symbol' in universe_history.columns else universe_history.index.get_level_values(1)
universe_history = universe_history.groupby([by_time, by_symbol]).agg(buycount=('isbuy', 'sum'), totalmentions=('direction', 'count'))
universe_history.index.names = ['time', 'symbol']
# Calculate and print universe metrics.
universe_size = universe_history.groupby(level='time').size()
print(f"Universe days: {len(universe_size)}")
print(f"Mean basket size per day: {universe_size.mean():.1f}")
factors = ['buycount', 'totalmentions']
for factor in factors:
    print(f'\n{universe_history[factor].describe()}')
# Extract unique assets chronologically and calculate the cumulative growth.
unique_by_date = universe_history.reset_index().sort_values('time').drop_duplicates(subset=['symbol']).set_index('time')
unique_by_date['cumulative_count'] = range(1, len(unique_by_date) + 1)
unique_by_date['cumulative_count'].plot(
    title='Cumulative Unique Assets Added to Universe', 
    ylabel='Total Unique Assets',
    marker='o'
);

### Daily Universe Prices

Fetch daily price history for every symbol that appears in the universe.

In [ ]:
# Extract unique assets
symbols = list(universe_history.index.get_level_values(1).unique())
# Fetch daily historical price metrics using the earliest timestamp available in the index.
history = qb.history(symbols, universe_history.index[0][0] - timedelta(1), qb.time, Resolution.DAILY)
history

### Align CNBC Signals And Returns

Build a joined table of factors and the future return.

In [ ]:
universe_history.index.names = ['time', 'symbol']
# Compute the forward open-to-open returns.
returns = history.open.unstack(0).pct_change().shift(-2).stack().rename('futurereturn')
returns.index.names = ['time', 'symbol']
# Join the dataframes and drop rows containing missing values.
dataset = universe_history[factors].join(returns, how='inner').dropna()
dataset.head()

### Analyze Relationships Between Factor and Future Returns

Create a grouped bar chart with standard error bars for each factor.

In [ ]:
for factor in factors:
    # Aggregate standard errors and observation counts across discrete group partitions.
    stats = dataset.groupby(factor)['futurereturn'].agg(['mean', 'sem', 'count']).reset_index()
    stats['sem'] = stats['sem'].fillna(0.0)
    plt.figure(figsize=(7, 4))
    plt.bar(stats[factor].astype(str), stats['mean'], yerr=stats['sem'], color='teal', alpha=0.8, capsize=5)
    plt.axhline(0, color='black', linewidth=1, alpha=0.4)
    plt.title(f'Mean Future Return by {factor}')
    plt.xlabel(f'Number of {factor}')
    plt.ylabel('Mean Future Return')
    for idx, row in stats.iterrows():
        y_pos = row['mean'] + (row['sem'] if row['mean'] >= 0 else -row['sem'])
        v_align = 'bottom' if row['mean'] >= 0 else 'top'
        plt.text(idx, y_pos, f"n={int(row['count'])}", ha='center', va=v_align, fontsize=9)
    plt.show()

Create a violin plot of the return density spread for each factor.

In [ ]:
# Map entire data dispersion structures over observed discrete frequency tiers.
for factor in factors:
    unique_vals = sorted(dataset[factor].unique())
    grouped_data = [dataset[dataset[factor] == val]['futurereturn'].values for val in unique_vals]
    # Construct descriptive geometric density maps.
    plt.figure(figsize=(7, 4))
    violins = plt.violinplot(grouped_data, showmeans=True, showmedians=False)
    for pc in violins['bodies']:
        pc.set_facecolor('cornflowerblue')
        pc.set_edgecolor('navy')
        pc.set_alpha(0.6)
    violins['cmaxes'].set_edgecolor('navy')
    violins['cmins'].set_edgecolor('navy')
    violins['cmeans'].set_edgecolor('tab:red')
    plt.axhline(0, color='black', linewidth=1, alpha=0.4)
    plt.title(f'Return Density Spread by {factor}')
    plt.xlabel(f'Number of {factor}')
    plt.ylabel('Future Return')
    plt.xticks(range(1, len(unique_vals) + 1), [str(v) for v in unique_vals])
    plt.show()